# Rich-Graph GNN Intrusion Detection: Version 2

## Study objective

This notebook develops the second version of the CIC IIoT 2025 intrusion-detection study. The objective is to determine whether graph neural networks trained on the same rich heterogeneous graph can outperform the classical tabular classification baselines.

The prediction task is binary:

- `0 = benign`
- `1 = attack`

Each observation represents one second of aggregated network activity for an IIoT device.

## GNN architectures

We will implement and compare five GNN architectures:

1. **GCN (Graph Convolutional Network)**
2. **GAT (Graph Attention Network)**
3. **GraphSAGE (Graph Sample and Aggregate)**
4. **R-GCN (Relational Graph Convolutional Network)**
5. **HGT (Heterogeneous Graph Transformer)**

All five models will receive the same rich graph, training split, validation split, test split, target definition, and evaluation metrics. GCN will serve as the standard graph-convolution baseline. GAT and GraphSAGE will provide alternative relation-agnostic neighborhood aggregation mechanisms, while R-GCN and HGT will explicitly use heterogeneous node or relation types. This controlled design helps us attribute performance differences to the GNN architecture rather than to different graph representations.

## Rich heterogeneous graph

The graph will represent multiple entity and relation types available in the dataset, including devices, behavioral states, IP-related information, ports, protocols, and temporal context. We will construct and validate this representation step by step before training any GNN.

The implementation will use PyTorch and a graph-learning library where appropriate. Each section will be developed interactively so that every tensor, graph object, model layer, training step, and evaluation result can be inspected and explained.

## Development workflow

We will proceed one stage at a time:

1. Inspect the raw benign and attack datasets.
2. Define leakage-safe training, validation, and test partitions.
3. Select and engineer the entities, relations, and node features.
4. Build and inspect the rich heterogeneous graph.
5. Convert the graph into PyTorch tensors.
6. Implement and understand one GNN architecture at a time.
7. Train, validate, and test each model under the same protocol.
8. Compare the five rich-GNN models with the classical baselines.

**Next step:** inspect the dataset columns and decide which columns can safely define graph entities, relations, node features, and labels.

## 1. Experimental protocol and learning goals

Before constructing the rich graph or training a neural network, we define one controlled protocol for all five GNN architectures. Each model will predict whether a 1-second IIoT traffic window is benign (`0`) or attack (`1`). GCN, GAT, GraphSAGE, R-GCN, and HGT will use the same observations, rich-graph entities, data partitions, target labels, and evaluation metrics.

The initial development split will use training, validation, and held-out test partitions with a fixed random seed. Model parameters are learned from training data, model selection is performed with validation data, and the test set is reserved for final evaluation. We will later add stricter time-based and device-based evaluations to investigate memorization and information leakage.

As we implement the project, we will learn the PyTorch concepts required by the actual code: tensors, shapes, data types, `nn.Module`, `forward()`, loss functions, gradients, optimizers, training mode, evaluation mode, and graph message passing.

**Neo4j checkpoint:** after constructing and validating the rich graph in Python, we will export its nodes and typed relationships for visualization in Neo4j. Neo4j will help us inspect the graph model; PyTorch Geometric will be used to train the GNNs.


In [60]:
from pathlib import Path # Path creates and manages filesystem paths in a platform-safe way.
import importlib.util # importlib.util lets us check whether a library is installed before importing it.
import random # random controls Python-based random operations.
import sys # sys provides information about the active Python interpreter.
import numpy as np # NumPy supports numerical arrays and reproducible numeric sampling.
import pandas as pd # pandas loads, organizes, and inspects the tabular CSV datasets.
import torch # PyTorch provides tensors, automatic differentiation, and neural-network tools.

# Use one shared seed to make random operations more reproducible.
SEED = 42
# Apply the seed to Python, NumPy, and PyTorch independently.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer an NVIDIA GPU when CUDA is available.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
# Otherwise, use Apple Metal acceleration when it is available on macOS.
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
# Fall back to the CPU when no supported accelerator is available.
else:
    DEVICE = torch.device("cpu")

# Check whether PyTorch Geometric is installed, and install it if it is not.
if importlib.util.find_spec("torch_geometric") is None:
    print("Installing PyTorch Geometric...")
    get_ipython().run_line_magic(
        "pip",
        "install torch-geometric --quiet --no-cache-dir"
        
    )
    PYG_AVAILABLE = "It was installed successfully."
else:
    print("PyTorch Geometric is already installed.")
    
import torch_geometric
PYG_AVAILABLE = True

# Display the environment information we need before building any graph.
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Pytorch Geometric: {torch_geometric.__version__}")
print(f"Compute device: {DEVICE}")
print(f"PyTorch Geometric installed: {PYG_AVAILABLE}")
print("Seeds used for reproducibility:", SEED)

PyTorch Geometric is already installed.
Python: 3.11.14
PyTorch: 2.11.0
Pytorch Geometric: 2.8.0.post1
Compute device: mps
PyTorch Geometric installed: True
Seeds used for reproducibility: 42


<h4>
PATH
</h4>

In [61]:
DATA_ROOT = Path(r"/Users/hectorcas13/Library/CloudStorage/Dropbox/University/PhD-Albany/CINF897-Independent_Study&Resarch/Graph GNN Comparisson/IoT_dataset/CIC IIoT dataset 2025")
BENIGN_PATH = DATA_ROOT / "Benign_data" / "benign_samples_1sec.csv"
ATTACK_PATH = DATA_ROOT / "Attack_samples" / "attack_samples_1sec.csv"

dataset_paths = {"benign": BENIGN_PATH, "attack": ATTACK_PATH}
path_check = pd.DataFrame([
    {"dataset": name, "path": str(path), "exists": path.exists()}
    for name, path in dataset_paths.items()
])
display(path_check)

if not path_check["exists"].all():
    raise FileNotFoundError("At least one dataset path is invalid. Update DATA_ROOT before continuing.")


,dataset,path,exists
0,benign,/Users/hectorcas13/Library/CloudStorage/Dropbo...,True
1,attack,/Users/hectorcas13/Library/CloudStorage/Dropbo...,True


## 2. First PyTorch concept: tensors

A tensor is PyTorch’s main data structure. A feature matrix is represented by a two-dimensional tensor: rows are observations or nodes and columns are features. Labels are stored in a separate tensor. The small example below is not a model; it is a safe way to inspect the structures that every later GNN will receive.


In [62]:
sample_x = torch.tensor(
    [
        [2.0, 1.0, 0.0],
        [25.0, 8.0, 6.0],
        [3.0, 1.0, 1.0],
    ],
    dtype=torch.float32,
)
sample_y = torch.tensor([0, 1, 0], dtype=torch.long)

print("Feature tensor:\n", sample_x)
print("Feature shape:", sample_x.shape)
print("Label tensor:", sample_y)
print("Label shape:", sample_y.shape)
print("Feature dtype:", sample_x.dtype)
print("Label dtype:", sample_y.dtype)

Feature tensor:
 tensor([[ 2.,  1.,  0.],
        [25.,  8.,  6.],
        [ 3.,  1.,  1.]])
Feature shape: torch.Size([3, 3])
Label tensor: tensor([0, 1, 0])
Label shape: torch.Size([3])
Feature dtype: torch.float32
Label dtype: torch.int64


### How to interpret the tensor example

`sample_x.shape == (3, 3)` means three observations and three features. `float32` is used for learnable numerical inputs. `sample_y.shape == (3,)` means one class label per observation, and `long` is the integer type commonly required for class-index labels. Later, the real feature tensors will describe graph nodes and `edge_index` tensors will describe their connections.

**Next step:** load only the dataset schema and a small sample, then decide which columns represent window features, node identities, relation types, and fields that must be excluded to prevent leakage.


In [63]:
# Load five rows from each dataset to inspect their structure.
benign_sample = pd.read_csv(BENIGN_PATH, nrows=5, on_bad_lines="warn")
attack_sample = pd.read_csv(ATTACK_PATH, nrows=5, on_bad_lines="warn")

# Show their dimensions.
print("Benign sample shape:", benign_sample.shape)
print("Attack sample shape:", attack_sample.shape)

Benign sample shape: (5, 94)
Attack sample shape: (5, 94)


In [64]:
# Create a numbered table with every dataset column.
columns_df = pd.DataFrame({
    "column_number": range(1, len(benign_sample.columns) + 1),
    "column_name": benign_sample.columns
})

display(columns_df)

,column_number,column_name
0,1,device_name
1,2,device_mac
2,3,label_full
3,4,label1
4,5,label2
...,...,...
89,90,network_ttl_std_deviation
90,91,network_window-size_avg
91,92,network_window-size_max
92,93,network_window-size_min


In [65]:
benign_all = pd.read_csv(BENIGN_PATH, on_bad_lines="warn")
attack_all = pd.read_csv(ATTACK_PATH, on_bad_lines="warn")

print(len(attack_all))
print(len(benign_all))

90391
136800


<h4>Combine the data</h4>

In [66]:
# Verify that benign and attack files have identical columns.
assert benign_all.columns.equals(attack_all.columns), \
    "Columns do not match between benign and attack datasets."

print("Columns match between benign and attack datasets.")
# Add a new column to each dataset to indicate whether the row is benign (0) or an attack (1).
benign_all["target"] = 0
attack_all["target"] = 1

# Combine both classes into one dataframe.
combined_df = pd.concat(
    [benign_all, attack_all], ignore_index=True,
    )

print("Combined shape:\n", combined_df.shape)
print("\nTarget distribution:")
display(combined_df["target"].value_counts())

Columns match between benign and attack datasets.
Combined shape:
 (227191, 95)

Target distribution:


target
0    136800
1     90391
Name: count, dtype: int64

<h4>Showing the columns to verify if formatting is needed</h4>

In [67]:
for col in combined_df.columns:
    print(f"{col}: {combined_df[col].iloc[0]} -> {combined_df[col].dtype}")

device_name: router -> str
device_mac: 28:87:ba:bd:c6:6c -> str
label_full: benign_whole-network3 -> str
label1: benign -> str
label2: benign -> str
label3: benign -> str
label4: benign -> str
timestamp: 2025-09-09T14:09:40.400000Z_2025-09-09T14:09:41.400000Z -> str
timestamp_start: 2025-09-09T14:09:40.400000Z -> str
timestamp_end: 2025-09-09T14:09:41.400000Z -> str
log_data-ranges_avg: 0.0 -> float64
log_data-ranges_max: 0.0 -> float64
log_data-ranges_min: 0.0 -> float64
log_data-ranges_std_deviation: 0.0 -> float64
log_data-types: [] -> str
log_data-types_count: 0 -> int64
log_interval-messages: 0.0 -> float64
log_messages_count: 0 -> int64
network_fragmentation-score: 0.0 -> float64
network_fragmented-packets: 0 -> int64
network_header-length_avg: 20.0 -> float64
network_header-length_max: 20.0 -> float64
network_header-length_min: 20.0 -> float64
network_header-length_std_deviation: 0.0 -> float64
network_interval-packets: 161.5 -> float64
network_ip-flags_avg: 2.0 -> float64
netwo

In [70]:
# time_start = pd.to_datetime(combined_df["timestamp_start"].str[:10], format="ISO8601", utc=True)
# time_end = pd.to_datetime(combined_df["timestamp_end"].str[:10],format="ISO8601", utc=True)

time_start = pd.to_datetime(combined_df["timestamp_start"], format="ISO8601", utc=True)
time_end = pd.to_datetime(combined_df["timestamp_end"],format="ISO8601", utc=True)

combined_df["timestamp_start_formatted"] = time_start
combined_df["timestamp_end_formatted"] = time_end

test_df = combined_df[
    ["timestamp_start_formatted", "timestamp_end_formatted"]
]
test_df.head(5)

,timestamp_start_formatted,timestamp_end_formatted
0,2025-09-09 14:09:40.400000+00:00,2025-09-09 14:09:41.400000+00:00
1,2025-09-09 14:09:41.400000+00:00,2025-09-09 14:09:42.400000+00:00
2,2025-09-09 14:09:42.400000+00:00,2025-09-09 14:09:43.400000+00:00
3,2025-09-09 14:09:43.400000+00:00,2025-09-09 14:09:44.400000+00:00
4,2025-09-09 14:09:44.400000+00:00,2025-09-09 14:09:45.400000+00:00


In [76]:
# Count missing values in each column.
missing_values = combined_df.isna().sum()

# Keep only columns that contain at least one missing value.
missing_summary = missing_values[
    missing_values > 0
].sort_values(ascending=False)

# Count completely duplicated rows.
duplicate_rows = combined_df.duplicated().sum()

print("Columns with missing values:", len(missing_summary))
display(missing_summary)

print("Completely duplicated rows:", duplicate_rows)

Columns with missing values: 0


Series([], dtype: int64)

Completely duplicated rows: 0


In [78]:
# Find existing label columns that may reveal the target.
label_columns = [
    column
    for column in combined_df.columns
    if column.startswith("label")
]

print("Potential leakage columns:", label_columns)

# Inspect their number of unique values.
label_summary = pd.DataFrame({
    "column": label_columns,
    "unique_values": [
        combined_df[column].nunique()
        for column in label_columns
    ],
    "examples": [
        combined_df[column].drop_duplicates().head(5).tolist()
        for column in label_columns
    ]
})

display(label_summary)

Potential leakage columns: ['label_full', 'label1', 'label2', 'label3', 'label4']


,column,unique_values,examples
0,label_full,937,"[benign_whole-network3, attack_ddos_syn-flood-..."
1,label1,2,"[benign, attack]"
2,label2,8,"[benign, ddos, dos, mitm, web]"
3,label3,61,"[benign, syn-flood-port-80, push-ack-flood-por..."
4,label4,84,"[benign, ddos_syn-flood-port-80, ddos_push-ack..."
